# 02 — Label Sanity Check

Visual confirmation that Phase 3 forward-return labels are correct.

**No logic lives here** — this notebook only reads the already-built
label and raw candle files, plots them, and lets a human eyeball whether
the labels make sense before trusting the numbers.

If any plot looks suspicious, inspect `src/labels/build_labels.py` and
run `pytest tests/test_labels.py -v`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

In [ ]:
INTERVAL = "1d"

candles = pd.read_parquet(f"data/raw/btc_usdt_{INTERVAL}.parquet")
labels = pd.read_parquet(f"data/processed/labels_{INTERVAL}.parquet")
manifest = json.loads(
    Path(f"data/processed/labels_{INTERVAL}.manifest.json").read_text()
)

# Merge on open_time for plotting — the join we test is clean.
df = candles.merge(labels, on="open_time", how="left")

print(f"Rows: {len(df)}")
print(f"Label columns: {[c for c in labels.columns if c != 'open_time']}")
for h, report in manifest["horizons"].items():
    bal = report["class_balance"]
    print(
        f"Horizon {h}: up={bal['up']} ({bal['up_pct']:.1f}%), "
        f"down={bal['down']} ({bal['down_pct']:.1f}%), "
        f"dead_zone={report['excluded_dead_zone_rows']}, "
        f"tail={report['excluded_tail_rows']}"
    )

## 1. Price with label colour overlay (60-day window)

Green = next candle goes up (label 1), Red = next candle goes down (label 0),
Grey = excluded (dead zone or tail).

In [ ]:
# Pick a 60-day window in the middle of the dataset.
mid = len(df) // 2
window = df.iloc[mid : mid + 60].copy()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(
    window["open_time"], window["close"],
    color="#333", lw=1, zorder=1,
)

# Colour-code each candle by its label.
for label_val, colour, name in [
    (1.0, "#2ecc71", "Up"),
    (0.0, "#e74c3c", "Down"),
]:
    mask = window["label_1"] == label_val
    ax.scatter(
        window.loc[mask, "open_time"],
        window.loc[mask, "close"],
        c=colour, s=30, zorder=2, label=name,
        edgecolors="none",
    )

# Excluded (NaN label).
excl = window["label_1"].isna()
ax.scatter(
    window.loc[excl, "open_time"],
    window.loc[excl, "close"],
    c="#bbb", s=30, zorder=2, label="Excluded",
    edgecolors="none",
)

ax.set_title(
    f"BTC/USDT {INTERVAL} — labels "
    f"(horizon=1, 60-candle window)",
    fontsize=12,
)
ax.set_ylabel("Close price (USDT)")
ax.legend(loc="upper left", framealpha=0.9)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 2. Class balance bar chart

In [ ]:
for h, report in manifest["horizons"].items():
    bal = report["class_balance"]
    categories = ["Up", "Down", "Dead zone", "Tail"]
    counts = [
        bal["up"], bal["down"],
        report["excluded_dead_zone_rows"],
        report["excluded_tail_rows"],
    ]
    colours = ["#2ecc71", "#e74c3c", "#bbb", "#888"]

    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(
        categories, counts, color=colours, edgecolor="white",
    )
    for bar, count in zip(bars, counts, strict=True):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 10,
            str(count), ha="center", va="bottom",
            fontsize=10,
        )
    dz_pct = report["dead_zone_pct"]
    ax.set_title(
        f"Class balance — horizon={h}, "
        f"dead_zone=\u00b1{dz_pct}%",
    )
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()

## 3. Forward return distribution with dead zone

In [ ]:
for h, report in manifest["horizons"].items():
    ret_col = f"fwd_return_{h}"
    dz = report["dead_zone_ratio"]
    returns = df[ret_col].dropna()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(
        returns * 100, bins=80,
        color="#3498db", edgecolor="white", alpha=0.8,
    )
    dz_label = f"{dz * 100:.2f}%"
    ax.axvline(
        dz * 100, color="#e74c3c", ls="--", lw=1.5,
        label=f"+dead zone ({dz_label})",
    )
    ax.axvline(
        -dz * 100, color="#e74c3c", ls="--", lw=1.5,
        label=f"\u2212dead zone ({dz_label})",
    )
    ax.set_title(
        f"Forward return distribution — horizon={h}",
    )
    ax.set_xlabel("Forward return (%)")
    ax.set_ylabel("Count")
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Summary stats.
    print(
        f"  Horizon {h}: "
        f"mean={returns.mean() * 100:.3f}%, "
        f"std={returns.std() * 100:.3f}%, "
        f"median={returns.median() * 100:.3f}%",
    )

## 4. Quick alignment sanity check

Spot-check that the label at row T matches what we compute by hand from
`close[T]` and `close[T+1]`.

In [ ]:
# Spot-check 5 random rows.
rng = np.random.default_rng(42)
n = len(df)
check_rows = sorted(rng.choice(n - 1, size=5, replace=False))

for t in check_rows:
    close_t = df["close"].iloc[t]
    close_t1 = df["close"].iloc[t + 1]
    expected_ret = close_t1 / close_t - 1.0
    actual_ret = df["fwd_return_1"].iloc[t]
    actual_lbl = df["label_1"].iloc[t]
    match = np.isclose(expected_ret, actual_ret)
    print(
        f"Row {t:>4d}: "
        f"close={close_t:>10.2f}, "
        f"close[T+1]={close_t1:>10.2f}, "
        f"ret={expected_ret:+.6f}, "
        f"actual={actual_ret:+.6f}, "
        f"label={actual_lbl}, "
        f"match={match}",
    )

all_ok = all(
    np.isclose(
        df["close"].iloc[t + 1] / df["close"].iloc[t] - 1.0,
        df["fwd_return_1"].iloc[t],
    )
    for t in check_rows
)
if all_ok:
    print("\n\u2705 All spot checks passed.")
else:
    print("\n\u274c Spot check FAILED \u2014 investigate!")